# Fine-tune_llama_3_with_ORPO
ORPO将传统的监督微调和偏好对齐阶段结合到一个过程中。这减少了训练所需的计算资源和时间。此外，实证结果表明，ORPO在各种模型大小和基准上都优于其他对齐方法。

## ORPO
- 虽然SFT有效地使模型适应所需的域，但它无意中增加了与首选答案一起生成不良答案的可能性。这就是为什么需要偏好对齐阶段来扩大首选输出和拒绝输出的可能性之间的差距。
- ORPO通过将指令调整和偏好对齐结合到一个单一的整体训练过程中，为这个问题提供了一个优雅的解决方案。ORPO修改了标准语言建模目标，将负对数似然损失与比值比（OR）项相结合。这种OR损失对被拒绝的响应进行了微弱的惩罚，同时对首选的响应进行了强烈的奖励，使模型能够同时学习目标任务并与人类的偏好保持一致。

## 使用ORPO微调LLaMA3

### 1. 安装并导入相关库

In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes wandb

In [ ]:
import gc
import os

import torch
import wandb
from datasets import load_dataset
from google.colab import userdata
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from trl import ORPOConfig, ORPOTrainer, setup_chat_format

wb_token = userdata.get('wandb')
wandb.login(key=wb_token)

如果您有最新的 GPU，您还应该能够使用 Flash Attention 库将默认的 Eager Attention 实现替换为更高效的实现

In [ ]:
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

### 2. 设置模型及参数

In [ ]:
# Model
base_model = "meta-llama/Meta-Llama-3-8B"
new_model = "OrpoLlama-3-8B"

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj',
                    'k_proj', 'q_proj', 'v_proj', 'o_proj']
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)
model, tokenizer = setup_chat_format(model, tokenizer)
model = prepare_model_for_kbit_training(model)

### 3. 加载数据集

In [ ]:
dataset_name = "mlabonne/orpo-dpo-mix-40k"
dataset = load_dataset(dataset_name, split="all")
dataset = dataset.shuffle(seed=42).select(range(100))


def format_chat_template(row):
    row["chosen"] = tokenizer.apply_chat_template(
        row["chosen"], tokenize=False)
    row["rejected"] = tokenizer.apply_chat_template(
        row["rejected"], tokenize=False)
    return row


dataset = dataset.map(
    format_chat_template,
    num_proc=os.cpu_count(),
)
dataset = dataset.train_test_split(test_size=0.01)

- 学习率：与传统的 SFT 甚至 DPO 相比，ORPO 的学习率非常低。8e-6 的这个值来自原始论文，大致对应于 SFT 学习率 1e-5 和 DPO 学习率 5e-6。我建议将其增加到 1e-6 左右以进行真正的微调。
- 其他参数（如批处理大小）设置为使用尽可能多的可用 VRAM（在此配置中为 ~20 GB）。
- 理想情况下，我们会训练 3-5 个周期的模型，但我们在这里坚持使用 1 个周期。

### 4. 训练模型

In [ ]:
orpo_args = ORPOConfig(
    learning_rate=8e-6,
    beta=0.1,
    lr_scheduler_type="linear",
    max_length=1024,
    max_prompt_length=512,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    num_train_epochs=1,
    evaluation_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    report_to="wandb",
    output_dir="./results/",
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    tokenizer=tokenizer,
)
trainer.train()
trainer.save_model(new_model)

### 5. 合并模型并上传Huggingface

In [ ]:
# Flush memory
del trainer, model
gc.collect()  # 调用了Python的垃圾收集器，用于释放不再使用的内存
torch.cuda.empty_cache()  

# Reload tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
model, tokenizer = setup_chat_format(model, tokenizer)

# Merge adapter with base model
model = PeftModel.from_pretrained(model, new_model)
model = model.merge_and_unload()

model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.push_to_hub(new_model, use_temp_dir=False)